# MNIST Neural Network Training

Training a fully connected neural network on the MNIST dataset using PyTorch.

## Imports and Setup

In [1]:
import torch
import torchvision
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
import multiprocessing
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("Torch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch version: 2.9.0+cu126
Device: cuda


## Dataset Class

In [2]:
class MNIST_dataset(Dataset):
    
    def __init__(self, data, partition="train"):
        print(f"\nLoading MNIST {partition} Dataset...")
        self.data = data
        self.partition = partition
        print(f"\tTotal Len.: {len(self.data)}\n{50*'-'}")

    def __len__(self):
        return len(self.data)
    
    def from_pil_to_tensor(self, image):
        return torchvision.transforms.ToTensor()(image)

    def __getitem__(self, idx):
        image = self.data[idx][0]
        image_tensor = self.from_pil_to_tensor(image)
        image_tensor = image_tensor.view(-1)

        label = torch.tensor(self.data[idx][1])
        label = F.one_hot(label, num_classes=10).float()

        return {"img": image_tensor, "label": label}

## Neural Network Architecture

In [3]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super(Net, self).__init__()
        self.linear1 = nn.Linear(784, 1024)
        self.relu1 = nn.ReLU()
        self.linear2 = nn.Linear(1024, 1024)
        self.relu2 = nn.ReLU()
        self.linear3 = nn.Linear(1024, 1024)
        self.relu3 = nn.ReLU()
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        out = self.relu1(self.linear1(x))
        out = self.relu2(self.linear2(out))
        out = self.relu3(self.linear3(out))
        out = self.classifier(out)
        return out

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Training and Evaluation Functions

In [4]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    train_loss, train_correct = 0, 0
    
    with tqdm(iter(dataloader), desc="Training", unit="batch") as tepoch:
        for batch in tepoch:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            labels = torch.argmax(labels, dim=1)
            pred = torch.argmax(outputs, dim=1)
            train_correct += pred.eq(labels).sum().item()
            train_loss += loss.item()

    train_loss /= len(dataloader.dataset)
    train_accuracy = 100. * train_correct / len(dataloader.dataset)
    return train_loss, train_accuracy


def evaluate(model, dataloader, criterion, device):
    model.eval()
    test_loss, test_correct = 0, 0
    
    with torch.no_grad():
        with tqdm(iter(dataloader), desc="Evaluating", unit="batch") as tepoch:
            for batch in tepoch:
                images = batch["img"].to(device)
                labels = batch["label"].to(device)

                outputs = model(images)
                test_loss += criterion(outputs, labels)

                labels = torch.argmax(labels, dim=1)
                pred = torch.argmax(outputs, dim=1)
                test_correct += pred.eq(labels).sum().item()

    test_loss /= len(dataloader.dataset)
    test_accuracy = 100. * test_correct / len(dataloader.dataset)
    return test_loss, test_accuracy


def train_model(model, train_loader, test_loader, criterion, optimizer, epochs, device, save_path="best_model.pt"):
    model.to(device)
    best_accuracy = -1
    best_epoch = 0
    
    print("\n---- Start Training ----")
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        
        train_loss, train_accuracy = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
        
        print(f"Train Loss: {train_loss:.6f} - Test Loss: {test_loss:.6f} - Train Accuracy: {train_accuracy:.2f}% - Test Accuracy: {test_accuracy:.2f}%")
        
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_epoch = epoch
            torch.save(model.state_dict(), save_path)
    
    print(f"\nBEST TEST ACCURACY: {best_accuracy:.2f}% in epoch {best_epoch + 1}")
    return best_accuracy, best_epoch

## Load and Prepare Data

In [5]:
train_set = torchvision.datasets.MNIST('.data/', train=True, download=True)
test_set = torchvision.datasets.MNIST('.data/', train=False, download=True)

print("Train images:", len(train_set))
print("Test images:", len(test_set))
print("\nSample image:", train_set[0][0])
print("Sample label:", train_set[0][1])
print("Sample label (one-hot):", F.one_hot(torch.tensor(train_set[0][1]), num_classes=10))

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.93MB/s]

100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]

100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 8.23MB/s]

Train images: 60000
Test images: 10000

Sample image: <PIL.Image.Image image mode=L size=28x28 at 0x7A8AF8283890>
Sample label: 5
Sample label (one-hot): tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])


## Create Datasets and DataLoaders

In [6]:
train_dataset = MNIST_dataset(train_set, partition="train")
test_dataset = MNIST_dataset(test_set, partition="test")

batch_size = 100
num_workers = multiprocessing.cpu_count() - 1
print(f"Num workers: {num_workers}")

train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=num_workers)


Loading MNIST train Dataset...
	Total Len.: 60000
--------------------------------------------------

Loading MNIST test Dataset...
	Total Len.: 10000
--------------------------------------------------
Num workers: 1


## Initialize Model and Training Configuration

In [7]:
num_classes = 10
net = Net(num_classes)
print(net)
print(f"\nTotal parameters: {count_parameters(net):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, weight_decay=1e-6, momentum=0.9)
epochs = 25

Net(
  (linear1): Linear(in_features=784, out_features=1024, bias=True)
  (relu1): ReLU()
  (linear2): Linear(in_features=1024, out_features=1024, bias=True)
  (relu2): ReLU()
  (linear3): Linear(in_features=1024, out_features=1024, bias=True)
  (relu3): ReLU()
  (classifier): Linear(in_features=1024, out_features=10, bias=True)
)

Total parameters: 2,913,290


## Train the Model

In [8]:
best_accuracy, best_epoch = train_model(
    net, 
    train_dataloader, 
    test_dataloader, 
    criterion, 
    optimizer, 
    epochs, 
    device,
    save_path="best_model.pt"
)


---- Start Training ----

Epoch 1/25


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 47.35batch/s]



Train Loss: 0.007234 - Test Loss: 0.002552 - Train Accuracy: 79.49% - Test Accuracy: 92.40%

Epoch 2/25


Evaluating: 100%|██████████| 100/100 [00:02<00:00, 38.07batch/s]

Train Loss: 0.002025 - Test Loss: 0.001538 - Train Accuracy: 94.10% - Test Accuracy: 95.33%

Epoch 3/25


...

Train Loss: 0.000008 - Test Loss: 0.000696 - Train Accuracy: 100.00% - Test Accuracy: 98.31%

Epoch 24/25



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 47.40batch/s]

Train Loss: 0.000007 - Test Loss: 0.000699 - Train Accuracy: 100.00% - Test Accuracy: 98.28%

Epoch 25/25



Evaluating: 100%|██████████| 100/100 [00:02<00:00, 48.19batch/s]

Train Loss: 0.000006 - Test Loss: 0.000705 - Train Accuracy: 100.00% - Test Accuracy: 98.32%

BEST TEST ACCURACY: 98.32% in epoch 25


## Load Best Model and Final Evaluation

In [9]:
net.load_state_dict(torch.load("best_model.pt"))
test_loss, test_accuracy = evaluate(net, test_dataloader, criterion, device)
print(f"\nFinal best accuracy: {test_accuracy:.2f}%")

Evaluating: 100%|██████████| 100/100 [00:02<00:00, 37.64batch/s]


Final best accuracy: 98.32%
